# RAG Formulary — LlamaIndex (Comparison Notebook)

**This is a comparison/learning artifact, not a proposed replacement for the project's RAG stack.** `02-Basic_RAG_Formulary.ipynb` is the project's real pipeline (LangChain + FAISS, table-aware chunking, model-keyed cache). This notebook asks the same question `ai-tutor-rag-system/notebooks/03-RAG_with_LlamaIndex.ipynb` asks of its own from-scratch predecessor: *what does the same task look like once a framework wires the plumbing?* — now applied to this project's PDF/table data instead of a CSV of blog posts.

It is **not** a recommendation to run two RAG frameworks in production side by side. Treat this as a side-by-side reference only.

## The one adaptation this comparison requires

`03-RAG_with_LlamaIndex.ipynb` builds its index with `VectorStoreIndex.from_documents(documents, transformations=[SentenceSplitter(...)])` — fine for prose, because the splitter chunks on sentence boundaries. Doing that here would be exactly the bug `langchain_03_table_aware.ipynb` fixed: a `SentenceSplitter` reading flattened table text has no idea a tier belongs to the drug name three words earlier, and would happily split them apart again.

So this notebook keeps the row-based extraction from `langchain_03_table_aware.ipynb` / `02-Basic_RAG_Formulary.ipynb` — one row, one chunk — and skips LlamaIndex's document/node-parsing step entirely: each row becomes a `TextNode` directly, and `VectorStoreIndex` is built **from nodes, not documents**. (LlamaIndex actually enforces this split in its own API: passing raw `Document` objects into the node-based constructor raises an error telling you to use `from_documents` instead — the framework agrees these are two different jobs.)

## 1. Setup

In [7]:
# Local Ollama LLM + local sentence-transformers embeddings, same as the rest of this
# project — no API keys needed.
import yaml
from pathlib import Path

config_path = Path("../../config/config.yaml")
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

pdf_dir = Path(f"../../{config['paths']['raw_data']}")
pdf_path = next(pdf_dir.glob("*.pdf"))
print(f"✅ Config loaded. PDF: {pdf_path.name}")

✅ Config loaded. PDF: Oscar_4T_NC_STND_Member_Doc__January_2026__as_of_11182025.pdf


## 2. Table-Aware Extraction → `TextNode` (not `Document`)

Same extractor as `langchain_03_table_aware.ipynb`, framework-agnostic this time (no LangChain `Document`, no LlamaIndex `Document` — plain dicts), so it can feed straight into a `TextNode` per row further down.

In [8]:
# Extract one record per drug row, keyed by its section/name/tier/requirements
import pdfplumber


def extract_drug_rows(path):
    rows = []
    current_section = None  # persists across pages until the next section header

    with pdfplumber.open(path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            for table in page.extract_tables():
                for row in table:
                    cells = [(c or "").strip().replace("\n", " ") for c in row]
                    non_empty = [c for c in cells if c]

                    if not non_empty:
                        continue
                    if "Drug Name" in cells:  # repeated table header row
                        continue
                    if len(non_empty) == 1:  # section header, e.g. "OPIOID ANALGESICS"
                        current_section = non_empty[0]
                        continue
                    if len(cells) < 4:  # page footer / legend row
                        continue

                    _, drug_name, tier, requirements = cells[:4]
                    if not drug_name:
                        continue

                    content = (
                        f"Section: {current_section or 'N/A'}\n"
                        f"Drug: {drug_name}\n"
                        f"Tier: {tier or 'N/A'}\n"
                        f"Requirements/Limits: {requirements or 'None'}"
                    )
                    rows.append({
                        "content": content,
                        "section": current_section,
                        "page": page_num,
                        "drug": drug_name,
                    })
    return rows


drug_rows = extract_drug_rows(pdf_path)
assert len(drug_rows) > 0, "No drug rows extracted — check the PDF's table layout hasn't changed."
print(f"✅ Extracted {len(drug_rows)} drug rows")

✅ Extracted 2394 drug rows


In [3]:
# One TextNode per row — already the final retrieval unit, no further splitting wanted.
from llama_index.core.schema import TextNode

nodes = [
    TextNode(
        text=row["content"],
        metadata={"section": row["section"], "page": row["page"], "drug": row["drug"]},
    )
    for row in drug_rows
]
print(f"✅ Built {len(nodes)} TextNodes (one per drug row, no sentence-splitting applied)")

✅ Built 2394 TextNodes (one per drug row, no sentence-splitting applied)


## 3. Index, with a Model-Keyed Cache

Same discipline as `02-Basic_RAG_Formulary.ipynb`, via LlamaIndex's own `StorageContext.persist()` / `load_index_from_storage()` instead of `FAISS.save_local`/`load_local`. The persist directory is named after the embedding model, so switching `embedding.model_name` in `config.yaml` points at a different (or missing) cache — never a stale one built by a different model.

In [9]:
from llama_index.core import VectorStoreIndex, StorageContext, load_index_from_storage
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(model_name=config["embedding"]["model_name"])

MODEL_TAG = config["embedding"]["model_name"].replace("/", "-")
PERSIST_DIR = Path(f"../../{config['paths']['processed_data']}/llamaindex_cache_{MODEL_TAG}_{config['embedding']['dimension']}d")

FORCE_REEMBED = False  # set True to rebuild the cache after changing extraction logic or the PDF
cache_exists = PERSIST_DIR.exists() and any(PERSIST_DIR.iterdir())

if cache_exists and not FORCE_REEMBED:
    print(f"Loading cached index from {PERSIST_DIR} ...")
    storage_context = StorageContext.from_defaults(persist_dir=str(PERSIST_DIR))
    index = load_index_from_storage(storage_context, embed_model=embed_model)
else:
    print(f"Embedding {len(nodes)} drug rows with {config['embedding']['model_name']} ...")
    index = VectorStoreIndex(nodes, embed_model=embed_model, show_progress=True)
    PERSIST_DIR.mkdir(parents=True, exist_ok=True)
    index.storage_context.persist(persist_dir=str(PERSIST_DIR))
    print(f"Cached index to {PERSIST_DIR}")

# Assert the COUNT, not just that something loaded — a stale or partial cache is a
# silent failure otherwise.
indexed_count = len(index.docstore.docs)
assert indexed_count == len(nodes), (
    f"Index has {indexed_count} nodes for {len(nodes)} rows — cache is stale or "
    "embedding collapsed. Set FORCE_REEMBED = True to rebuild."
)
print(f"✅ knowledge base: {indexed_count} rows indexed via {config['embedding']['model_name']}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1725.26it/s]


Loading cached index from ../../data/processed/NC/2026/formulary/llamaindex_cache_sentence-transformers-all-MiniLM-L6-v2_384d ...
✅ knowledge base: 2394 rows indexed via sentence-transformers/all-MiniLM-L6-v2


## 4. Query Engine

`index.as_query_engine(llm=...)` bundles embedding the question, retrieving top-k rows, building the augmentation prompt, and calling the LLM — the same collapse `03-RAG_with_LlamaIndex.ipynb` demonstrates, now backed by the local Ollama model this project already uses instead of a cloud API.

In [10]:
from llama_index.llms.ollama import Ollama

llm = Ollama(
    model=config["llm"]["model"],
    temperature=config["llm"]["temperature"],
    request_timeout=300.0,
    # Without this, llama_index queries the model's native max context (131072 for
    # llama3.1) and asks Ollama to allocate a KV cache sized for it on every call —
    # tens of GB for a CPU-only run, regardless of how small the actual RAG prompt
    # is. This corpus's retrieved rows + question never approach 4096 tokens.
    context_window=4096,
)

query_engine = index.as_query_engine(llm=llm, similarity_top_k=config["retrieval"]["top_k"])
print(f"✅ Query engine ready: {config['llm']['model']}")

✅ Query engine ready: llama3.1


In [11]:
# Same question as 02-Basic_RAG_Formulary.ipynb, for a direct answer comparison.
response = query_engine.query(
    "What is the tier and quantity limit for acetaminophen with codeine "
    "solution 120-12 mg/5ml and tablet 300-15 mg?"
)
print("Answer:", response)

print(f"\n=== Retrieved (top_k={config['retrieval']['top_k']}) ===")
for rank, node in enumerate(response.source_nodes, start=1):
    print(f"┌─ Rank {rank} | score {node.score:.4f}")
    print("│  " + node.node.get_content().replace("\n", " "))
    print("└" + "─" * 60)

Answer: The acetaminophen with codeine solution 120-12 mg/5ml is Tier 1 and has a quantity limit of 2700 ml every 30 days. The acetaminophen with codeine tablet 300-15 mg is Tier 1 and has a quantity limit of 390 tablets every 30 days.

=== Retrieved (top_k=3) ===
┌─ Rank 1 | score 0.7619
│  Section: OPIOID ANALGESICS Drug: acetaminophen w/ codeine soln 120-12 mg/5ml Tier: Tier 1 Requirements/Limits: QL (2700 ml every 30 days); Subject to initial 5- day limit for 19 and younger; 7-day initial limit for all other ages
└────────────────────────────────────────────────────────────
┌─ Rank 2 | score 0.7081
│  Section: OPIOID ANALGESICS Drug: acetaminophen w/ codeine tab 300-15 mg Tier: Tier 1 Requirements/Limits: QL (390 tabs every 30 days); Subject to initial 5- day limit for 19 and younger; 7-day initial limit for all other ages
└────────────────────────────────────────────────────────────
┌─ Rank 3 | score 0.6991
│  Section: OPIOID ANALGESICS Drug: acetaminophen w/ codeine tab 300-60 mg

In [12]:
# A question genuinely outside this corpus (a drug formulary has no dental benefit
# info) — mirrors the "Llama 4" out-of-corpus check in 03-RAG_with_LlamaIndex.ipynb.
response_oos = query_engine.query("What is the copay for a routine dental cleaning?")
print("Answer:", response_oos)

Answer: There is no information provided about the copay for a routine dental cleaning.


## 🔑 Key Takeaways

- **The framework collapse is real but must be steered.** `as_query_engine()` replaces ~15 lines of manual retrieval + prompt templating from `02-Basic_RAG_Formulary.ipynb` with one call — but only after deliberately building `TextNode`s directly and bypassing `VectorStoreIndex.from_documents()`'s default sentence-splitting, or the table-row bug this project already fixed would come back.
- **Caching ports across frameworks, not just providers.** `StorageContext.persist()` / `load_index_from_storage()` is LlamaIndex's version of the same model-keyed cache used with `FAISS.save_local`/`load_local` — same rule (a cache belongs to exactly one embedding model), different API.
- **Less visible ≠ less correct, but it is less inspectable.** `02-Basic_RAG_Formulary.ipynb` shows FAISS L2 distances and lets you see the exact prompt template; here, `response.source_nodes[i].score` and the augmentation prompt are available but one layer further from view.
- **This notebook is a comparison, not a migration.** The project's actual RAG stack stays LangChain + FAISS (`langchain_03_table_aware.ipynb`, `02-Basic_RAG_Formulary.ipynb`); running two frameworks side by side in production would be dependency weight with no corresponding benefit unless there's a concrete reason to switch.